# Week 3 · Day 5: Gradient descent from scratch

**Course:** Labs 06–07: Gradient descent and scikit-learn

The teaching question is **“Is this source's absolute G magnitude below 4?”** Class 1 means yes. This is a brightness label, not a verified stellar type. Only `bp_rp` (color) enters the model; magnitude and parallax are used to construct the answer and must stay out of the inputs.

Day 1 saves a source-disjoint 60/20/20 split. Days 2–6 explore training and validation; Day 7 evaluates the reserved test set. Absolute magnitudes are uncorrected for dust, and the archive sample may not represent all Gaia sources.

## Setup

[Prerequisites and setup explanation](week3_prerequisites.ipynb). Run cells from top to bottom. Run Day 1 first. This notebook reads its saved data.

In [ ]:
from pathlib import Path
import sys
import importlib
import numpy as np
import pandas as pd
from IPython.display import display

# Jupyter may start in the project folder or in its notebooks folder.
ROOT = Path.cwd()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
if not (ROOT / 'src/classification.py').is_file():
    raise RuntimeError('Open Jupyter from gaia-explorer or gaia-explorer/notebooks.')
sys.path.insert(0, str(ROOT))
from src import classification as data, visualize
# Refresh the data loader if this kernel imported an older version.
data = importlib.reload(data)
visualize = importlib.reload(visualize)  # Pick up updated plot labels and layout.

DATA_PATH = None  # None uses the shared W2 cache; a missing cache is downloaded.
OUTPUT = ROOT / 'data/processed/week3'
OUTPUT.mkdir(parents=True, exist_ok=True)
pd.set_option('display.max_columns', 20)


## Load the Day 1 data

These lines select the saved splits. `X` contains color and `y` contains the brightness label.

In [ ]:
prepared_path = OUTPUT / 'learning_data.csv'
if not prepared_path.is_file():
    raise FileNotFoundError('Run Week 3 Day 1 first to create learning_data.csv.')

# Keep source IDs as text so their digits are not rounded.
gaia = pd.read_csv(prepared_path, dtype={'source_id': 'string'})
train = gaia.loc[gaia['split'] == 'train'].copy()
validation = gaia.loc[gaia['split'] == 'validation'].copy()
test = gaia.loc[gaia['split'] == 'test'].copy()
feature_columns = ['bp_rp']
X_train = train[feature_columns]
y_train = train['is_bright']
X_val = validation[feature_columns]
y_val = validation['is_bright']
print('Training rows:', len(train), '| Validation rows:', len(validation))


## 1. Reuse the equations from Days 2 and 4

These short definitions are repeated here so this notebook runs on its own after Day 1.

In [ ]:
def sigmoid(z):
    """Turn a score into a probability, safely even for very large scores."""
    # This equals 1 / (1 + exp(-z)); logaddexp avoids overflow.
    return np.exp(-np.logaddexp(0, -np.asarray(z, dtype=float)))

def compute_cost(X, y, w, b, lambda_=0.0):
    """Mean logistic loss plus L2 regularization; bias b is not penalized."""
    m = len(y)
    z = X @ w + b
    # Stable form of -y*log(sigmoid(z)) - (1-y)*log(1-sigmoid(z)).
    loss = np.mean(np.logaddexp(0, z) - y * z)
    regularization = lambda_ * np.sum(w ** 2) / (2 * m)
    return loss + regularization


## 2. Calculate gradients

The W2 pattern remains: calculate errors, then weight and bias gradients. W3 uses sigmoid predictions. L2 adds λw/m to the weight gradient.

In [ ]:
def compute_gradient(X, y, w, b, lambda_=0.0):
    m = len(y)
    predictions = sigmoid(X @ w + b)
    errors = predictions - y
    dw = X.T @ errors / m + lambda_ * w / m
    db = np.mean(errors)
    return dw, db


## 3. Write the training loop

Start at zero, calculate both gradients, then update both parameters. Save the initial cost and the cost after every step.

In [ ]:
def gradient_descent(X, y, learning_rate=0.1, iterations=4000, lambda_=1.0):
    w = np.zeros(X.shape[1])
    b = 0.0
    history = [compute_cost(X, y, w, b, lambda_)]
    for step in range(iterations):
        dw, db = compute_gradient(X, y, w, b, lambda_)
        w = w - learning_rate * dw
        b = b - learning_rate * db
        history.append(compute_cost(X, y, w, b, lambda_))
    return w, b, np.array(history)


## 4. Scale and train

Expected output: learned weights, bias, and a falling cost curve. Scaling uses training rows only.

In [ ]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
y = y_train.to_numpy()
w, b, history = gradient_descent(X, y)
print('w:', w, '| b:', b)
visualize.plot_binary_curves(np.arange(len(history)), {'training cost': history}, xlabel='Iteration', ylabel='Regularized cost', title='Scratch logistic training')

## 5. Compare with scikit-learn

Use the same data and L2 strength. With this unweighted objective, lambda=1/C. Expected output: similar probabilities and validation losses; small optimization differences are normal.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import log_loss
reference = LogisticRegression(C=1.0, max_iter=5000, tol=1e-10)
reference.fit(X, y)
scratch_p = sigmoid(X_val_scaled @ w + b)
sklearn_p = reference.predict_proba(X_val_scaled)[:, 1]
print('Mean probability difference:', np.mean(abs(scratch_p - sklearn_p)))
print('Scratch validation loss:', log_loss(y_val, scratch_p))
print('Sklearn validation loss:', log_loss(y_val, sklearn_p))
display(pd.DataFrame({'feature': feature_columns, 'scratch_weight': w, 'sklearn_weight': reference.coef_[0]}))

## 6. Experiment with learning rate

Expected output: three curves for the same 400-step budget. Small steps may learn slowly; large steps can oscillate. Describe what your curves show.

In [ ]:
curves = {}
for rate in [0.001, 0.1, 10.0]:
    _, _, costs = gradient_descent(X, y, learning_rate=rate, iterations=400)
    curves[f'rate={rate}'] = costs
visualize.plot_binary_curves(np.arange(401), curves, xlabel='Iteration', ylabel='Cost', title='Learning rate experiment')

## Reflection questions and answers

1. **What changed from W2 gradient descent?** In W2 linear regression, the score `X @ w + b` was the predicted magnitude, and we reduced squared-error cost. In W3 classification, that score goes through `sigmoid` to become a probability of class 1, and we reduce logistic loss. Both training loops calculate gradients and repeatedly update `w` and `b`; the predictions and cost being optimized have changed.

2. **Why might the scratch and scikit-learn solvers disagree?** They use different optimization procedures and may stop at slightly different points, so small differences in weights and probabilities are normal. If the difference is large, check whether scratch gradient descent had enough iterations, whether its learning rate is stable, whether both models saw the same scaled training rows, and whether the regularization strengths match. For the unweighted L2 objective used here, `lambda_ = 1.0` in the scratch code corresponds to `C = 1.0` in this scikit-learn comparison. A coding error in the cost or gradient can also cause disagreement. Compare the validation probabilities and losses, not just the weights.

3. **Why avoid fitting the scaler on validation rows?** `scaler.fit_transform(X_train)` learns the training colors' mean and standard deviation. `scaler.transform(X_val)` then applies those *same* values to validation colors. Fitting a new scaler on validation would give it a different measurement scale from the one the model learned. Fitting on training and validation together would also let validation information influence training. Keeping validation separate makes it a fair check on new stars.


## References

Course labs: `../ML-coursework/W3/Optional Labs/` in the workspace.

[ESA Gaia DR3](https://www.cosmos.esa.int/web/gaia/dr3) · [Extinction guidance](https://www.cosmos.esa.int/web/gaia/edr3-extinction-law) · [Scikit-learn logistic regression](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LogisticRegression.html)